# Notebook 07 — Regularization Experiments

**Research question:** How do Ridge, Lasso, and ElasticNet change the behaviour of a linear regression model on this customer-value prediction problem?

**Rules for this notebook:**
- Use the SAME frozen train/test split as Notebook 06 (RANDOM_STATE=42, TEST_SIZE=0.20)
- Use the SAME 5-fold CV strategy
- Do NOT evaluate against the test set in this notebook — that is reserved for Notebook 08
- This notebook is for **understanding model behaviour**, not selecting the winner

| Section | Topic |
|---|---|
| 1 | Setup and frozen split |
| 2 | Ridge — theory and L2 penalty |
| 3 | Ridge — alpha grid search (CV only) |
| 4 | Ridge — coefficient path |
| 5 | Ridge — coefficient stability vs OLS |
| 6 | Lasso — theory and L1 penalty |
| 7 | Lasso — alpha grid search (CV only) |
| 8 | Lasso — sparsity and feature selection |
| 9 | ElasticNet — theory, alpha and l1_ratio |
| 10 | ElasticNet — grid over alpha and l1_ratio |
| 11 | Summary comparison across all three |

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import src as _src_pkg
if 'ecommerce_customer_value_prediction' not in sys.modules:
    sys.modules['ecommerce_customer_value_prediction'] = _src_pkg
import warnings; warnings.filterwarnings('ignore')
import json, math, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
from ecommerce_customer_value_prediction.config import PROCESSED_DATA_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR
MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
from ecommerce_customer_value_prediction.models.train import (
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET_COL,
    build_pipeline, create_model, train, save_model, load_model, MODEL_PATHS,
    FINAL_MODEL_PATH, FINAL_METADATA_PATH, SUPPORTED_MODELS,
)
from ecommerce_customer_value_prediction.models.evaluate import evaluate
from ecommerce_customer_value_prediction.models.predict import predict
from sklearn.model_selection import train_test_split, KFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
CUSTOMER_CSV = PROCESSED_DATA_DIR / 'customer_dataset.csv'
df = pd.read_csv(CUSTOMER_CSV)
df['Customer ID'] = df['Customer ID'].astype('Int64')
RANDOM_STATE = 42; TEST_SIZE = 0.20; CV_FOLDS = 5
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df[TARGET_COL].copy()
y_raw = df['future_revenue'].copy()
y_bins = pd.cut(y_raw, bins=[-0.01, 0.01, 50, 500, float('inf')], labels=['zero','low','medium','high'])
X_train, X_test, y_train, y_test, y_raw_train, y_raw_test = train_test_split(
    X, y, y_raw, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_bins)
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
def rmse(y_true, y_pred): return np.sqrt(mean_squared_error(y_true, y_pred))
scoring = {'mae': make_scorer(mean_absolute_error), 'rmse': make_scorer(rmse), 'r2': make_scorer(r2_score)}
print(f'Dataset: {df.shape} | Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Modules loaded. RANDOM_STATE={RANDOM_STATE}')

---
## Section 2 — Ridge Regression: L2 Regularisation

### OLS vs Ridge objective

**OLS:**
$$\min_{\beta} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Ridge:**
$$\min_{\beta} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} \beta_j^2$$

The second term $\alpha \sum \beta_j^2$ is the **L2 penalty** (also called Tikhonov regularisation).

### What alpha does

| alpha | Effect |
|---|---|
| 0 | No penalty → identical to OLS |
| Small (0.001) | Mild shrinkage; close to OLS |
| Medium (1.0) | Moderate shrinkage |
| Large (1000) | Strong shrinkage; all coefficients approach zero |
| → ∞ | All coefficients = 0; model predicts the mean |

### Why Ridge stabilises correlated features

For two perfectly correlated features X1 and X2:
- OLS: assigns any split between β1 and β2 (unstable)
- Ridge: prefers equal distribution of coefficient across X1 and X2 (stable)

This is because the L2 penalty grows with β², so large individual coefficients are penalised.
Ridge cannot zero out coefficients — it can only shrink them.

In [ ]:
# Manual Ridge demonstration with a tiny example
from sklearn.linear_model import Ridge, LinearRegression
import numpy as np

# Two correlated features: x2 ≈ x1
np.random.seed(42)
n = 200
x1 = np.random.randn(n)
x2 = x1 + 0.1 * np.random.randn(n)   # almost perfectly correlated with x1
y_toy = 2 * x1 + 0.5 * np.random.randn(n)

X_toy = np.column_stack([x1, x2])

ols = LinearRegression().fit(X_toy, y_toy)
ridge_mild = Ridge(alpha=1.0).fit(X_toy, y_toy)
ridge_strong = Ridge(alpha=100.0).fit(X_toy, y_toy)

print('Toy example: y = 2*x1 + noise  (x2 ≈ x1 — correlated)')
print()
print(f'           beta_x1    beta_x2    sum_betas')
print(f'OLS      : {ols.coef_[0]:>8.4f}   {ols.coef_[1]:>8.4f}   {ols.coef_.sum():>8.4f}')
print(f'Ridge α=1: {ridge_mild.coef_[0]:>8.4f}   {ridge_mild.coef_[1]:>8.4f}   {ridge_mild.coef_.sum():>8.4f}')
print(f'Ridge α=100: {ridge_strong.coef_[0]:>8.4f} {ridge_strong.coef_[1]:>8.4f}   {ridge_strong.coef_.sum():>8.4f}')
print()
print('Observation: OLS assigns wild ± split; Ridge distributes evenly')
print('Both sum to ≈2.0 (the true coefficient), but Ridge is stable.')

---
## Section 3 — Ridge: Alpha Grid Search (CV Only)

We sweep alpha over a logarithmic grid and record training + CV performance.

In [ ]:
alphas_ridge = np.logspace(-3, 4, 50)   # 0.001 → 10000
ridge_results = []

for alpha in alphas_ridge:
    pipe = build_pipeline(estimator=create_model('ridge', alpha=alpha))
    cv_scores = cross_validate(pipe, X_train, y_train, cv=kf, scoring=scoring)
    pipe_fit = build_pipeline(estimator=create_model('ridge', alpha=alpha))
    pipe_fit.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, pipe_fit.predict(X_train))
    ridge_results.append({
        'alpha': alpha,
        'train_mae': train_mae,
        'cv_mae': cv_scores['test_mae'].mean(),
        'cv_rmse': cv_scores['test_rmse'].mean(),
        'cv_r2': cv_scores['test_r2'].mean(),
    })

ridge_df = pd.DataFrame(ridge_results)
best_ridge_idx = ridge_df['cv_r2'].idxmax()
best_ridge_alpha = ridge_df.loc[best_ridge_idx, 'alpha']
print(f'Best Ridge alpha (by CV R²): {best_ridge_alpha:.4f}')
print(f'  CV R²  : {ridge_df.loc[best_ridge_idx, "cv_r2"]:.4f}')
print(f'  CV RMSE: {ridge_df.loc[best_ridge_idx, "cv_rmse"]:.4f}')
print(f'  CV MAE : {ridge_df.loc[best_ridge_idx, "cv_mae"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, metric, label in zip(axes, ['cv_mae', 'cv_rmse', 'cv_r2'],
                              ['CV MAE (log)', 'CV RMSE (log)', 'CV R²']):
    ax.semilogx(ridge_df['alpha'], ridge_df[metric], 'steelblue', lw=2)
    ax.axvline(best_ridge_alpha, color='red', linestyle='--', label=f'Best α={best_ridge_alpha:.3f}')
    ax.set_xlabel('alpha (log scale)')
    ax.set_ylabel(label)
    ax.set_title(f'Ridge: {label} vs alpha')
    ax.legend(fontsize=9)
plt.suptitle('Ridge Regularisation: CV Performance vs Alpha', y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/ridge_alpha_cv.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: reports/figures/ridge_alpha_cv.png')

---
## Section 4 — Ridge Coefficient Path

The **coefficient path** shows how each coefficient changes as alpha increases.

Key observations to look for:
1. At alpha → 0: coefficients approach OLS values (potentially large/unstable)
2. At alpha → ∞: all coefficients shrink toward zero
3. Ridge NEVER sets any coefficient to exactly zero (unlike Lasso)
4. Correlated features tend to converge to equal values as alpha increases

In [ ]:
alphas_path = np.logspace(-2, 4, 80)
coef_paths_ridge = {f: [] for f in NUMERIC_FEATURES}

for alpha in alphas_path:
    pipe = build_pipeline(estimator=create_model('ridge', alpha=alpha))
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps['regressor'].coef_
    for i, f in enumerate(NUMERIC_FEATURES):
        coef_paths_ridge[f].append(coefs[i])

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(NUMERIC_FEATURES)))
for f, color in zip(NUMERIC_FEATURES, colors):
    ax.semilogx(alphas_path, coef_paths_ridge[f], lw=2, label=f, color=color)
ax.axvline(best_ridge_alpha, color='black', linestyle='--', lw=1.5, label=f'Best α={best_ridge_alpha:.3f}')
ax.axhline(0, color='gray', linestyle='-', lw=0.5)
ax.set_xlabel('alpha (log scale)')
ax.set_ylabel('Coefficient (standardised features)')
ax.set_title('Ridge Coefficient Path (numeric features only)')
ax.legend(loc='right', fontsize=9)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/ridge_coef_path.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: reports/figures/ridge_coef_path.png')

### What the path reveals

1. **No sparsity:** At no alpha value does any coefficient reach exactly zero. Ridge shrinks but never eliminates.

2. **Correlated features converge:** `monetary`, `frequency`, `unique_products` are collinear. At high alpha they converge to similar (small) values — Ridge distributes the shared credit equally.

3. **Recency_days stays negative:** The negative direction of `recency_days` (higher recency = less future revenue) is preserved across all alpha values. The sign is robust.

4. **The best alpha is the one that minimises CV RMSE** — not training RMSE. The training RMSE always decreases with lower alpha (less constraint = better fit to training data). Only CV exposes the true generalisation.

---
## Section 5 — Ridge Coefficient Stability vs OLS

Recall from Notebook 06 that OLS coefficients were highly unstable across bootstrap resamples.
Here we repeat the same experiment with Ridge to show the stabilisation effect.

In [ ]:
N_RESAMPLES = 30
rng = np.random.default_rng(RANDOM_STATE)
tracked = ['monetary', 'avg_basket_value', 'frequency', 'unique_products', 'recency_days']
ols_coefs_r = {f: [] for f in tracked}
ridge_coefs_r = {f: [] for f in tracked}

for _ in range(N_RESAMPLES):
    idx = rng.integers(0, len(X_train), size=len(X_train))
    X_rs, y_rs = X_train.iloc[idx], y_train.iloc[idx]
    for model_name, store, alpha_val in [('linear', ols_coefs_r, None), ('ridge', ridge_coefs_r, best_ridge_alpha)]:
        est = create_model(model_name) if model_name == 'linear' else create_model('ridge', alpha=alpha_val)
        pipe = build_pipeline(estimator=est)
        pipe.fit(X_rs, y_rs)
        coef_arr = pipe.named_steps['regressor'].coef_[:len(NUMERIC_FEATURES)]
        for i, f in enumerate(NUMERIC_FEATURES):
            if f in tracked:
                store[f].append(coef_arr[i])

print(f'Coefficient stability: OLS vs Ridge (alpha={best_ridge_alpha:.4f})')
print(f'  {"Feature":<22} | {"OLS std":>9} | {"Ridge std":>9} | {"Reduction":>10}')
print('  ' + '-'*60)
for f in tracked:
    ols_std = np.std(ols_coefs_r[f])
    ridge_std = np.std(ridge_coefs_r[f])
    reduction = (1 - ridge_std / (ols_std + 1e-9)) * 100
    print(f'  {f:<22} | {ols_std:>9.4f} | {ridge_std:>9.4f} | {reduction:>9.1f}%')

---
## Section 6 — Lasso Regression: L1 Regularisation

**Lasso objective:**
$$\min_{\beta} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^{p} |\beta_j|$$

The penalty is the **L1 norm** (sum of absolute values of coefficients).

### Why L1 produces sparsity (exact zeros)

The L2 penalty ($\beta^2$) is smooth and differentiable. At $\beta=0$ the derivative is 0, so the optimiser gently pushes coefficients toward zero but never reaches exactly zero unless data strongly supports it.

The L1 penalty ($|\beta|$) is **not differentiable at $\beta=0$**. The subgradient at zero spans $[-\alpha, +\alpha]$. The optimiser finds that setting a coefficient to exactly zero satisfies the optimality condition if the data's signal for that feature is small enough relative to $\alpha$.

**Geometric intuition:**
- Ridge constraint region: sphere (smooth — solutions on surface but rarely on axes)
- Lasso constraint region: diamond (corners on the axes — solutions often land on corners where one or more $\beta_j = 0$)

### Key difference from Ridge

| Property | Ridge | Lasso |
|---|---|---|
| Coefficient at alpha → ∞ | 0 (but never exactly) | 0 (exactly) |
| Feature selection | No | Yes (implicitly) |
| Handles correlated features | Keeps both, shrinks equally | Keeps one, zeros the other |
| Differentiable objective | Yes (closed form) | No (coordinate descent) |

In [ ]:
alphas_lasso = np.logspace(-4, 1, 60)   # 0.0001 → 10
lasso_results = []

for alpha in alphas_lasso:
    pipe = build_pipeline(estimator=create_model('lasso', alpha=alpha))
    cv_scores = cross_validate(pipe, X_train, y_train, cv=kf, scoring=scoring)
    pipe_fit = build_pipeline(estimator=create_model('lasso', alpha=alpha))
    pipe_fit.fit(X_train, y_train)
    coefs_num = pipe_fit.named_steps['regressor'].coef_[:len(NUMERIC_FEATURES)]
    n_nonzero = int((np.abs(pipe_fit.named_steps['regressor'].coef_) > 1e-6).sum())
    lasso_results.append({
        'alpha': alpha,
        'cv_mae': cv_scores['test_mae'].mean(),
        'cv_rmse': cv_scores['test_rmse'].mean(),
        'cv_r2': cv_scores['test_r2'].mean(),
        'n_nonzero': n_nonzero,
    })

lasso_df = pd.DataFrame(lasso_results)
best_lasso_idx = lasso_df['cv_r2'].idxmax()
best_lasso_alpha = lasso_df.loc[best_lasso_idx, 'alpha']
print(f'Best Lasso alpha (by CV R²): {best_lasso_alpha:.5f}')
print(f'  CV R²       : {lasso_df.loc[best_lasso_idx, "cv_r2"]:.4f}')
print(f'  CV RMSE     : {lasso_df.loc[best_lasso_idx, "cv_rmse"]:.4f}')
print(f'  Non-zero coef: {int(lasso_df.loc[best_lasso_idx, "n_nonzero"])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].semilogx(lasso_df['alpha'], lasso_df['cv_r2'], 'steelblue', lw=2)
axes[0].axvline(best_lasso_alpha, color='red', linestyle='--', label=f'Best α={best_lasso_alpha:.4f}')
axes[0].set_xlabel('alpha'); axes[0].set_ylabel('CV R²'); axes[0].set_title('Lasso: CV R² vs alpha'); axes[0].legend()
axes[1].semilogx(lasso_df['alpha'], lasso_df['n_nonzero'], 'darkorange', lw=2)
axes[1].axvline(best_lasso_alpha, color='red', linestyle='--')
axes[1].set_xlabel('alpha'); axes[1].set_ylabel('Non-zero coefficients'); axes[1].set_title('Lasso: Sparsity vs alpha')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/lasso_alpha_cv.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: reports/figures/lasso_alpha_cv.png')

---
## Section 8 — Lasso Coefficient Path and Sparsity

In [ ]:
alphas_path_lasso = np.logspace(-4, 0, 80)
coef_paths_lasso = {f: [] for f in NUMERIC_FEATURES}

for alpha in alphas_path_lasso:
    pipe = build_pipeline(estimator=create_model('lasso', alpha=alpha))
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps['regressor'].coef_
    for i, f in enumerate(NUMERIC_FEATURES):
        coef_paths_lasso[f].append(coefs[i])

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(NUMERIC_FEATURES)))
for f, color in zip(NUMERIC_FEATURES, colors):
    ax.semilogx(alphas_path_lasso, coef_paths_lasso[f], lw=2, label=f, color=color)
ax.axvline(best_lasso_alpha, color='black', linestyle='--', lw=1.5, label=f'Best α={best_lasso_alpha:.4f}')
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('alpha (log scale)'); ax.set_ylabel('Coefficient'); ax.set_title('Lasso Coefficient Path')
ax.legend(loc='right', fontsize=9)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/lasso_coef_path.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: reports/figures/lasso_coef_path.png')

In [ ]:
# Examine which features are zeroed at best Lasso alpha
pipe_lasso_best = build_pipeline(estimator=create_model('lasso', alpha=best_lasso_alpha))
pipe_lasso_best.fit(X_train, y_train)
lasso_coefs = pipe_lasso_best.named_steps['regressor'].coef_[:len(NUMERIC_FEATURES)]
lasso_coef_df = pd.DataFrame({'feature': NUMERIC_FEATURES, 'coefficient': lasso_coefs})
lasso_coef_df['is_zero'] = lasso_coef_df['coefficient'].abs() < 1e-6
print(f'Lasso (alpha={best_lasso_alpha:.5f}) coefficient summary:')
print(lasso_coef_df.to_string(index=False))
print()
zeroed = lasso_coef_df[lasso_coef_df['is_zero']]['feature'].tolist()
kept   = lasso_coef_df[~lasso_coef_df['is_zero']]['feature'].tolist()
print(f'Features ZEROED by Lasso: {zeroed}')
print(f'Features KEPT  by Lasso: {kept}')
print()
print('Why zero does NOT mean the variable is unimportant:')
print('  When two features are collinear (e.g. frequency and unique_products),')
print('  Lasso may zero one of them arbitrarily. The zeroed feature still')
print('  carries real-world signal — it was simply redundant given its correlate.')
print('  Do not interpret a zero coefficient as "this feature does not matter."')

---
## Section 9 — ElasticNet: Combined L1 + L2 Regularisation

**ElasticNet objective:**
$$\min_{\beta} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \left[ \frac{(1-\rho)}{2} \sum_j \beta_j^2 + \rho \sum_j |\beta_j| \right]$$

Where:
- $\alpha$ controls overall regularisation strength
- $\rho$ (sklearn: `l1_ratio`) controls the L1/L2 mix:
  - $\rho = 0$ → pure Ridge (L2 only)
  - $\rho = 1$ → pure Lasso (L1 only)
  - $0 < \rho < 1$ → blend

### sklearn parameterisation

```python
ElasticNet(alpha=0.1, l1_ratio=0.5)
```

`alpha` in sklearn's ElasticNet = overall regularisation strength (scales both L1 and L2).
`l1_ratio` in sklearn = ρ above.

### When ElasticNet is useful

1. **Many correlated features:** Lasso arbitrarily zeros one of a correlated pair. ElasticNet (with L2 component) tends to shrink correlated features together, keeping more of them.
2. **Groups of features:** Ridge keeps all features; Lasso selects one per group; ElasticNet keeps the group with moderate shrinkage.
3. **p >> n problems:** Lasso can select at most n features; ElasticNet can select more.

In [ ]:
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
alphas_en = np.logspace(-4, 1, 25)
en_results = []

for l1_ratio in l1_ratios:
    for alpha in alphas_en:
        pipe = build_pipeline(estimator=create_model('elasticnet', alpha=alpha, l1_ratio=l1_ratio))
        cv_scores = cross_validate(pipe, X_train, y_train, cv=kf, scoring=scoring)
        pipe_fit = build_pipeline(estimator=create_model('elasticnet', alpha=alpha, l1_ratio=l1_ratio))
        pipe_fit.fit(X_train, y_train)
        n_nz = int((np.abs(pipe_fit.named_steps['regressor'].coef_) > 1e-6).sum())
        en_results.append({
            'alpha': alpha, 'l1_ratio': l1_ratio,
            'cv_r2': cv_scores['test_r2'].mean(),
            'cv_rmse': cv_scores['test_rmse'].mean(),
            'n_nonzero': n_nz,
        })

en_df = pd.DataFrame(en_results)
best_en_idx = en_df['cv_r2'].idxmax()
best_en_alpha = en_df.loc[best_en_idx, 'alpha']
best_en_l1 = en_df.loc[best_en_idx, 'l1_ratio']
print(f'Best ElasticNet: alpha={best_en_alpha:.5f}  l1_ratio={best_en_l1}')
print(f'  CV R²  : {en_df.loc[best_en_idx, "cv_r2"]:.4f}')
print(f'  CV RMSE: {en_df.loc[best_en_idx, "cv_rmse"]:.4f}')

In [ ]:
# Heatmap: CV R² across alpha and l1_ratio
pivot = en_df.pivot_table(index='l1_ratio', columns='alpha', values='cv_r2')
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, annot=False, fmt='.3f', cmap='RdYlGn', ax=ax, cbar_kws={'label': 'CV R²'})
ax.set_xlabel('alpha'); ax.set_ylabel('l1_ratio')
ax.set_title('ElasticNet: CV R² heatmap (alpha × l1_ratio)')
ax.set_xticklabels([f'{v:.3f}' for v in sorted(en_df["alpha"].unique())], rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/elasticnet_heatmap.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: reports/figures/elasticnet_heatmap.png')

---
## Section 11 — Summary: OLS vs Ridge vs Lasso vs ElasticNet

This section compares development-phase CV metrics. Test-set evaluation in Notebook 08.

In [ ]:
# OLS CV baseline
ols_pipe = build_pipeline(estimator=create_model('linear'))
ols_cv = cross_validate(ols_pipe, X_train, y_train, cv=kf, scoring=scoring)

ridge_cv_best = cross_validate(build_pipeline(estimator=create_model('ridge', alpha=best_ridge_alpha)),
                               X_train, y_train, cv=kf, scoring=scoring)
lasso_cv_best = cross_validate(build_pipeline(estimator=create_model('lasso', alpha=best_lasso_alpha)),
                               X_train, y_train, cv=kf, scoring=scoring)
en_cv_best = cross_validate(build_pipeline(estimator=create_model('elasticnet', alpha=best_en_alpha, l1_ratio=best_en_l1)),
                            X_train, y_train, cv=kf, scoring=scoring)

print('Development CV Summary (training set only — test NOT used):')
print(f'  {"Model":<20} | {"CV R²":>8} | {"CV RMSE":>8}')
print('  ' + '-'*45)
for label, cv in [('OLS', ols_cv), ('Ridge (best)', ridge_cv_best),
                  ('Lasso (best)', lasso_cv_best), ('ElasticNet (best)', en_cv_best)]:
    print(f'  {label:<20} | {cv["test_r2"].mean():>8.4f} | {cv["test_rmse"].mean():>8.4f}')
print()
print('NOTE: These are development metrics from cross-validation on the training set.')
print('Final test-set evaluation is in Notebook 08.')